## Prepare data
Load a multirep session from a `session.pkl` file, session directory, or `.tar.gz` archive.

In [ ]:
import sys, importlib.util
from pathlib import Path

# VS Code injects __vsc_ipynb_file__ with the notebook's absolute path.
_nb = globals().get("__vsc_ipynb_file__") or ""
if not _nb:
    raise RuntimeError("Select the project venv kernel: .venv/bin/python")

REPO_ROOT = Path(_nb).resolve().parent.parent  # analysis/ -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import directly — skips analysis/__init__.py (avoids eager matplotlib/torch load)
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_loader = _load_module("multirep_loader", REPO_ROOT / "analysis" / "multirep_loader.py")
load_session              = _loader.load_session
load_session_from_tarball = _loader.load_session_from_tarball

# Point to a session directory, session.pkl, or .tar.gz archive.
# Set to None to auto-select the latest .tar.gz in multirepData/.
SESSION_PATH = REPO_ROOT / "experiment" / "data" / "multirepData" / "<tarfile>"
SESSION_PATH = None

_data_dir = REPO_ROOT / "experiment" / "data" / "multirepData"

if SESSION_PATH is None:
    _tarballs = sorted(_data_dir.glob("*.tar.gz"), key=lambda p: p.stat().st_mtime)
    if not _tarballs:
        raise FileNotFoundError(f"No .tar.gz files found in {_data_dir}")
    SESSION_PATH = _tarballs[-1]
    print(f"Auto-selected: {SESSION_PATH.name}")

path = Path(SESSION_PATH)
if path.suffixes[-2:] == [".tar", ".gz"]:
    session = load_session_from_tarball(path)
else:
    session = load_session(path)

print(f"Session:    {session.session_id}")
print(f"Preset:     {session.preset_name}")
print(f"Timestamp:  {session.session_timestamp}")
print(f"Tasks:      {session.n_tasks}")
print(f"Datasets:   {session.datasets}")
print(f"Rep rows:   {len(session.reputation_timeline)}")
print(f"Acc rows:   {len(session.global_accuracy)}")

## Access top-level data

In [2]:
rep     = session.reputation_timeline   # one row per (task, user)
acc     = session.global_accuracy       # one row per (task, round)
tasks   = session.tasks                 # list of per-task metadata dicts
preset  = session.preset

## Reputation timeline
One row per `(task_index, user)`. Contains pre/post reputation values, selection decision, and confidence.

**Key columns:** `task_index`, `dataset`, `task_type`, `user_name`, `behavior`, `was_selected`, `was_cached`, `tr_pre`, `tr_post`, `tr_all_post`, `gir_pre`, `gir_post`, `q_pre`, `q_post`, `balance_pre`, `balance_post`, `selection_score`, `confidence`, `k`

In [3]:
rep

,task_index,dataset,task_type,fingerprint,user_name,user_address,guid,behavior,was_selected,was_cached,...,tr_all_post,gir_post,q_post,q_all_post,balance_post,total_contrib_post,confidence,k,running_c_mean,m2
0,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced A,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,296baa1b-ecd2-e910-43b2-ed0f556c8caf,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
1,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,Balanced B,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,607f92d4-7776-41d1-9fd8-d1544b28d35e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
2,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 1,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,bee79562-79c0-d807-1571-44d0b752438e,honest,True,False,...,{5: 0.0},0.000000,0.416667,{5: 0.4166666666666667},0.0,0.0,0.000000,0,0.000000,0.000000
3,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,MNIST Expert 2,0x90F79bf6EB2c4f870365E785982E1f101E93b906,ac193986-b823-1eb7-af58-2839a4c10301,honest,False,False,...,{5: 0.0},0.005556,0.000000,{5: 0.0},0.0,0.0,0.166667,1,0.000000,0.000000
4,0,mnist,5,44c360e506610e5e3961cc8b39fe0ffac98f2c3114d1b7...,CIFAR Expert 1,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,627ce64d-fccf-0d4b-3c15-a6105eefa35a,honest,False,False,...,{5: 0.0},0.000000,0.000000,{5: 0.0},0.0,0.0,0.166667,1,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,9,cifar-10,6,09bb951fff00e366242178a5c73b9afa749e2321ff53b8...,Cross Mal-MNIST Honest-CIFAR,0x14dC79964da2C08b23698B3D3cc7Ca32193d9955,b7fd4edd-361d-6858-0a0e-37ce12be516c,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.036698,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.0,0.0,0.281125,2,0.273545,0.000816
116,9,cifar-10,6,09bb951fff00e366242178a5c73b9afa749e2321ff53b8...,Malicious Both,0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f,31c4d68b-1f18-071e-dd3b-499b715548f9,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.036698,0.166667,"{5: 0.16666666666666666, 1: 0.0, 2: 0.0, 3: 0....",0.0,0.0,0.281125,2,0.273545,0.000816
117,9,cifar-10,6,09bb951fff00e366242178a5c73b9afa749e2321ff53b8...,Free-rider Both,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,34eac3e3-3c1c-9ae8-841a-94fcb2cf42e5,freerider,True,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.189557,0.750000,"{5: 0.75, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0...",0.0,0.0,0.283177,2,0.222222,0.000448
118,9,cifar-10,6,09bb951fff00e366242178a5c73b9afa749e2321ff53b8...,Passive A,0xBcd4042DE499D14e55001CcbB24a551F3b954096,520d0604-83ca-5769-ec11-2bcb5483c0b2,honest,False,False,...,"{5: 0.016259297660024083, 1: 0.0, 2: 0.0, 3: 0...",0.093378,0.916667,"{5: 0.3333333333333333, 1: 0.0, 2: 0.0, 3: 0.0...",0.0,0.0,0.092517,2,0.646259,0.104413


## Global accuracy
One row per `(task_index, round)`. Aggregated from per-task run data.

**Key columns:** `task_index`, `dataset`, `round`, `objective_global_accuracy`, `objective_global_loss`, `reward_pool`, `punishment_pool`

In [4]:
acc

,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool,task_index,dataset
0,0,0.000000,0.1343,2.300781,1000000000000000000,0,0,mnist
1,1,1.899709,0.5110,2.248047,666666666666666667,666666666666666664,0,mnist
2,2,1.748052,0.6643,2.085938,333333333333333334,444444444444444444,0,mnist
3,3,1.763069,0.7374,1.683594,1,666666666666666669,0,mnist
4,0,0.000000,0.1000,2.308594,1000000000000000000,0,1,cifar-10
5,1,1.978051,0.1000,2.302734,666666666666666667,0,1,cifar-10
6,2,2.971001,0.1000,2.302734,333333333333333334,0,1,cifar-10
7,3,2.915983,0.1000,2.302734,1,0,1,cifar-10
8,0,0.000000,0.1173,2.302734,1000000000000000000,0,2,mnist
9,1,1.735809,0.1905,2.308594,666666666666666667,499999999999999998,2,mnist


## Per-task run data
Each task in `session.tasks` may embed the full tables from its individual `.pkl` run.
Use `session.get_task_run_data(task_index)` to retrieve them, or iterate with `session.iter_run_data()`.

**Tables per task:** `global`, `users`, `votes`, `receipts`, `contributions`, `warnings`, `task_rep_calc`, `trs`

In [5]:
import pandas as pd

rows = []
for t in tasks:
    rd = t.get("run_data")
    has_data = rd is not None and any(
        (hasattr(df, "empty") and not df.empty) for df in rd.values()
    )
    rows.append({
        "task_index": t["task_index"],
        "dataset":    t["dataset"],
        "task_type":  t["task_type"],
        "was_cached": t["was_cached"],
        "has_run_data": has_data,
    })

pd.DataFrame(rows)

,task_index,dataset,task_type,was_cached,has_run_data
0,0,mnist,5,False,True
1,1,cifar-10,6,False,True
2,2,mnist,5,False,True
3,3,cifar-10,6,False,True
4,4,mnist,5,False,True
5,5,cifar-10,6,False,True
6,6,mnist,5,False,True
7,7,cifar-10,6,False,True
8,8,mnist,5,False,True
9,9,cifar-10,6,False,True


In [6]:
# Inspect a specific task
TASK_INDEX = 1

rd = session.get_task_run_data(TASK_INDEX)
if rd is None:
    print(f"Task {TASK_INDEX} has no embedded run data.")
else:
    global_df       = rd.get("global",        pd.DataFrame())
    users_df        = rd.get("users",          pd.DataFrame())
    votes_df        = rd.get("votes",          pd.DataFrame())
    receipts_df     = rd.get("receipts",       pd.DataFrame())
    contributions_df= rd.get("contributions",  pd.DataFrame())
    task_rep_df     = rd.get("task_rep_calc",  pd.DataFrame())
    warnings_df     = rd.get("warnings",       pd.DataFrame())
    trs_df          = rd.get("trs",            pd.DataFrame())

    for name, df in [("global", global_df), ("users", users_df), ("votes", votes_df),
                     ("receipts", receipts_df), ("contributions", contributions_df),
                     ("task_rep_calc", task_rep_df), ("warnings", warnings_df), ("trs", trs_df)]:
        print(f"{name}: {df.shape}")

global: (4, 7)
users: (20, 17)
votes: (60, 11)
receipts: (68, 5)
contributions: (15, 17)
task_rep_calc: (5, 13)
warnings: (0, 0)
trs: (1, 5)


In [7]:
global_df

,experiment_id,round,round_time,objective_global_accuracy,objective_global_loss,reward_pool,punishment_pool
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,0.000000,0.1,2.308594,1000000000000000000,0
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,1.978051,0.1,2.302734,666666666666666667,0
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,2.971001,0.1,2.302734,333333333333333334,0
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,3,2.915983,0.1,2.302734,1,0


In [8]:
users_df

,experiment_id,round,user_number,state,behavior,role,grs,address,id,subjective_personal_accuracy,subjective_personal_loss,subjective_global_accuracy,subjective_global_loss,round_reputation_assigned,reward_delta,is_reward,merged
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,5,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,NaN,NaN,NaN,NaN,NaN,NaN,None,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,4,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,45676c35-64c6-4443-a85d-4e96102c1af8,NaN,NaN,NaN,NaN,NaN,NaN,None,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,0,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,aceed2eb-306f-4b68-a70e-7ce1961ada5a,NaN,NaN,NaN,NaN,NaN,NaN,None,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,1,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,dde06797-fb6f-4887-993c-082e094c760a,NaN,NaN,NaN,NaN,NaN,NaN,None,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,0,11,active,Attitude.Honest,Attitude.Honest,1000000000000000000,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,9d06ad3d-40c6-4b67-850a-18384eda9b6b,NaN,NaN,NaN,NaN,NaN,NaN,None,None
5,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,5,active,Attitude.Honest,Attitude.Honest,1058139534883720776,None,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,0.000000,2.322266,0.1003,2.304688,4.000000e+18,5.813953e+16,True,True
6,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,4,active,Attitude.Honest,Attitude.Honest,1069767441860464936,None,45676c35-64c6-4443-a85d-4e96102c1af8,0.333333,2.255859,0.0879,2.304688,3.000000e+18,6.976744e+16,True,True
7,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,0,active,Attitude.Honest,Attitude.Honest,1073643410852713422,None,aceed2eb-306f-4b68-a70e-7ce1961ada5a,0.147500,2.291484,0.1446,2.294922,4.000000e+18,7.364341e+16,True,True
8,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,1,active,Attitude.Honest,Attitude.Honest,1073643410852713422,None,dde06797-fb6f-4887-993c-082e094c760a,0.100000,2.299688,0.1001,2.300781,4.000000e+18,7.364341e+16,True,True
9,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,11,active,Attitude.Honest,Attitude.Honest,1058139534883720776,None,9d06ad3d-40c6-4b67-850a-18384eda9b6b,0.120000,2.305156,0.0934,2.304688,2.000000e+18,5.813953e+16,True,True


In [9]:
votes_df

,experiment_id,round,giver_id,receiver_id,giver_address,receiver_address,vote_feedback_score,vote_prev_accuracy,vote_prev_loss,vote_accuracy,vote_loss
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,45676c35-64c6-4443-a85d-4e96102c1af8,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,1,NaN,232,0,229
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,aceed2eb-306f-4b68-a70e-7ce1961ada5a,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,1,NaN,232,0,230
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,dde06797-fb6f-4887-993c-082e094c760a,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,1,NaN,232,0,229
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,9d06ad3d-40c6-4b67-850a-18384eda9b6b,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,1,NaN,232,4000,228
4,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,45676c35-64c6-4443-a85d-4e96102c1af8,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,1,NaN,233,5000,218
5,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,45676c35-64c6-4443-a85d-4e96102c1af8,aceed2eb-306f-4b68-a70e-7ce1961ada5a,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,1,NaN,233,3667,225
6,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,45676c35-64c6-4443-a85d-4e96102c1af8,dde06797-fb6f-4887-993c-082e094c760a,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,1,NaN,233,5000,227
7,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,45676c35-64c6-4443-a85d-4e96102c1af8,9d06ad3d-40c6-4b67-850a-18384eda9b6b,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,-1,NaN,233,0,232
8,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,aceed2eb-306f-4b68-a70e-7ce1961ada5a,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,1,1000.0,231,1025,230
9,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,aceed2eb-306f-4b68-a70e-7ce1961ada5a,45676c35-64c6-4443-a85d-4e96102c1af8,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0,1000.0,231,750,230


In [10]:
receipts_df

,experiment_id,round,tx_type,tx_hash,gas_used
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,slot,9042359e8e3f4f1b28fce8bb8e31461d4eca516f0daef5...,50837
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,slot,69c1551bc7343529dc173a216ad0eb70f22758cca71dc9...,50837
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,slot,af1df7268c401ea67a0a65ee6c74fa53b1cda2a8b9b91c...,50837
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,slot,e3e527037c64c5b2450d880e03d7985a2ac3ba2d9a58e1...,50837
4,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,slot,626d4d061906d79aa49b0ce0348eddc95f56a2e2968150...,50837
...,...,...,...,...,...
63,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,4,exit,d7ff3cbcc3b99e6f445719c7d21945cbbca9fa3eacea6d...,56148
64,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,4,exit,fff6aa2b67c3fbec42a76ffbd1a57a289bee850a8860ec...,58712
65,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,4,exit,9c873ee0ea3bdf4f36615ed6b65d76c2f7642d6071ca2c...,56476
66,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,4,exit,4493e79717485a088a3a7bfe53be5d61777bf159295e8f...,53912


In [11]:
contributions_df

,experiment_id,round,user_id,user_address,contribution_score,user_mad_avg,current_excluded_values,current_accepted_values,current_mad_median,current_mad_value,current_mad_max_deviation,prev_avg_round_val_after_mad,previous_excluded_values,previous_accepted_values,previous_mad_median,previous_mad_value,previous_mad_max_deviation
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,5,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,174418604651162323,230.000000,"[218, 231]","[230, 230]",230.0,0.5,0.815419,231.25,[233],"[232, 231, 231, 231]",231.0,0.0,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,4,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,209302325581394804,229.750000,[],"[229, 230, 230, 230]",230.0,0.0,NaN,231.25,[233],"[232, 231, 231, 231]",231.0,0.0,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,220930232558140260,229.666667,[225],"[230, 229, 230]",229.5,0.5,0.815419,231.25,[233],"[232, 231, 231, 231]",231.0,0.0,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,1,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,220930232558140260,229.666667,[227],"[229, 230, 230]",229.5,0.5,0.815419,231.25,[233],"[232, 231, 231, 231]",231.0,0.0,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,1,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,174418604651162323,230.000000,"[228, 232]","[230, 230]",230.0,1.0,1.630838,231.25,[233],"[232, 231, 231, 231]",231.0,0.0,None
5,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,5,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,0,229.750000,[],"[229, 230, 230, 230]",230.0,0.0,NaN,229.75,[228],"[229, 230, 230, 230]",230.0,0.0,None
6,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,4,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,0,229.750000,[],"[229, 230, 230, 230]",230.0,0.0,NaN,229.75,[228],"[229, 230, 230, 230]",230.0,0.0,None
7,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,0,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,333333333333333314,229.666667,[228],"[229, 230, 230]",229.5,0.5,0.815419,229.75,[228],"[229, 230, 230, 230]",230.0,0.0,None
8,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,1,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,333333333333333314,229.666667,[228],"[229, 230, 230]",229.5,0.5,0.815419,229.75,[228],"[229, 230, 230, 230]",230.0,0.0,None
9,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,2,11,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,333333333333333314,229.666667,[228],"[229, 230, 230]",229.5,0.5,0.815419,229.75,[228],"[229, 230, 230, 230]",230.0,0.0,None


In [12]:
import pandas as pd

all_contributions = pd.concat([
    rd.get("contributions", pd.DataFrame()).assign(task_index=task_index)
    for task_index, dataset, rd in session.iter_run_data()
    if rd.get("contributions") is not None and not rd["contributions"].empty
], ignore_index=True)

all_contributions["contribution_score"] /= 1e18

user_names = rep[["user_address", "user_name"]].drop_duplicates()
all_contributions = all_contributions.merge(user_names, on="user_address", how="left")
all_contributions["user"] = all_contributions["user_name"].fillna(all_contributions["user_id"].astype(str))

pivot = all_contributions.pivot_table(
    index="user",
    columns="task_index",
    values="contribution_score",
    aggfunc="mean"
).rename_axis(None, axis=1)

selected_counts = all_contributions.groupby("task_index")["user"].nunique().rename("# selected")
pivot.loc["# selected"] = selected_counts
pivot

/tmp/ipykernel_54950/509570012.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_contributions = pd.concat([


,0,1,2,3,4,5,6,7,8,9
user,,,,,,,,,,
Balanced A,NaN,0.323643,NaN,0.454848,NaN,NaN,NaN,NaN,0.460450,NaN
Balanced B,NaN,0.323643,NaN,0.342489,1.239256,1.065476,1.0,NaN,0.642177,NaN
CIFAR Expert 1,NaN,0.069767,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.624994
CIFAR Expert 2,NaN,-0.025194,0.898305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cross Honest-MNIST Mal-CIFAR,0.960317,NaN,1.700565,NaN,0.641116,NaN,NaN,NaN,NaN,0.249982
Cross Mal-MNIST Honest-CIFAR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,NaN,0.124993
Free-rider Both,NaN,NaN,NaN,-1.000000,NaN,-0.833333,NaN,NaN,NaN,NaN
MNIST Expert 1,NaN,NaN,-1.000000,NaN,-1.000000,0.255952,NaN,NaN,NaN,0.250045
MNIST Expert 2,NaN,NaN,NaN,-0.066167,NaN,0.255952,NaN,NaN,NaN,NaN


In [13]:
task_rep_df

,experiment_id,user_id,address,task_type,k,running_c_mean,m2,global_task_rep,global_integrity_rep,task_rep_delta,final_grs,positive_votes,total_votes
0,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,474cc5e4-fa2f-4d3e-9f20-99f1f9b7fcc9,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,6,0,0.692829,0.0,0.023094,0.200000,None,None,None,None
1,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,45676c35-64c6-4443-a85d-4e96102c1af8,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,6,0,0.764120,0.0,0.025471,0.168056,None,None,None,None
2,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,aceed2eb-306f-4b68-a70e-7ce1961ada5a,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,6,0,0.925618,0.0,0.030854,0.200000,None,None,None,None
3,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,dde06797-fb6f-4887-993c-082e094c760a,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,6,0,0.925618,0.0,0.030854,0.200000,None,None,None,None
4,cifar-10-loss_only-1-0.1-1-1.0-False-{0e7fddb9...,9d06ad3d-40c6-4b67-850a-18384eda9b6b,0x71bE63f3384f5fb98995898A86B02Fb2426c5788,6,0,0.914544,0.0,0.030485,0.168056,None,None,None,None


In [ ]:
import pandas as pd

all_task_rep = pd.concat([
    rd.get("task_rep_calc", pd.DataFrame()).assign(task_index=task_index)
    for task_index, dataset, rd in session.iter_run_data()
    if rd.get("task_rep_calc") is not None and not rd["task_rep_calc"].empty
], ignore_index=True)

all_task_rep["task_rep_delta"] = pd.to_numeric(all_task_rep["task_rep_delta"], errors="coerce") / 1e18

user_names = rep[["user_address", "user_name"]].drop_duplicates()
all_task_rep = all_task_rep.merge(user_names, left_on="address", right_on="user_address", how="left")
all_task_rep["user"] = all_task_rep["user_name"].fillna(all_task_rep["user_id"].astype(str))

pivot = all_task_rep.pivot_table(
    index="user",
    columns="task_index",
    values="task_rep_delta",
    aggfunc="mean"
).rename_axis(None, axis=1)

selected_counts = all_task_rep.groupby("task_index")["user"].nunique().rename("# selected")
pivot.loc["# selected"] = selected_counts
pivot

## Merge users with their votes (per task)
Attach receiver behavior/role to each vote row.

In [14]:
if votes_df.empty or users_df.empty:
    print("votes or users is empty for this task.")
else:
    votes_with_receiver = votes_df.merge(
        users_df[["experiment_id", "round", "address", "behavior", "role"]],
        left_on=["experiment_id", "round", "receiver_address"],
        right_on=["experiment_id", "round", "address"],
        how="left"
    ).rename(columns={
        "behavior": "receiver_behavior",
        "role":     "receiver_role",
    }).drop(columns=["address"])

    votes_with_receiver

## Reputation analysis across all tasks

In [15]:
# Selection frequency per user
rep[rep["was_selected"]].groupby(["user_name", "behavior"])["task_index"].count().rename("times_selected").reset_index()

,user_name,behavior,times_selected
0,Balanced A,honest,6
1,Balanced B,honest,3
2,CIFAR Expert 1,honest,4
3,CIFAR Expert 2,honest,5
4,Cross Honest-MNIST Mal-CIFAR,honest,5
5,Cross Mal-MNIST Honest-CIFAR,malicious,4
6,Free-rider Both,freerider,4
7,MNIST Expert 1,honest,3
8,MNIST Expert 2,honest,4
9,Malicious Both,malicious,4


In [16]:
# Mean final TR per behavior group
rep.groupby("behavior")[["tr_post", "gir_post", "q_post", "balance_post"]].mean().round(4)

,tr_post,gir_post,q_post,balance_post
behavior,,,,
freerider,0.0069,0.1174,0.4833,0.0000
honest,0.0261,0.1507,0.3778,1.5922
malicious,0.0034,0.0039,0.5000,0.0000


In [17]:
# Filter reputation timeline to a specific task
rep[rep["task_index"] == TASK_INDEX]

,task_index,dataset,task_type,fingerprint,user_name,user_address,guid,behavior,was_selected,was_cached,...,tr_all_post,gir_post,q_post,q_all_post,balance_post,total_contrib_post,confidence,k,running_c_mean,m2
12,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Balanced A,0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266,296baa1b-ecd2-e910-43b2-ed0f556c8caf,honest,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.200000,0.000000,"{5: 0.4166666666666667, 1: 0.0, 2: 0.0, 3: 0.0...",0.325258,0.0,0.166667,1,0.925618,0.0
13,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Balanced B,0x70997970C51812dc3A010C7d01b50e0d17dc79C8,607f92d4-7776-41d1-9fd8-d1544b28d35e,honest,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.200000,0.000000,"{5: 0.4166666666666667, 1: 0.0, 2: 0.0, 3: 0.0...",0.325258,0.0,0.166667,1,0.925618,0.0
14,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,MNIST Expert 1,0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC,bee79562-79c0-d807-1571-44d0b752438e,honest,True,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.416667,"{5: 0.4166666666666667, 1: 0.0, 2: 0.0, 3: 0.0...",0.000000,0.0,0.000000,0,0.000000,0.0
15,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,MNIST Expert 2,0x90F79bf6EB2c4f870365E785982E1f101E93b906,ac193986-b823-1eb7-af58-2839a4c10301,honest,True,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.005556,0.416667,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.0,0.000000,0,0.000000,0.0
16,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,CIFAR Expert 1,0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65,627ce64d-fccf-0d4b-3c15-a6105eefa35a,honest,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.168056,0.000000,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.069767,0.0,0.166667,1,0.764120,0.0
17,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,CIFAR Expert 2,0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc,dddb0f74-089e-1653-cb0f-fc271205848a,honest,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.200000,0.000000,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.0,0.166667,1,0.692829,0.0
18,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Cross Honest-MNIST Mal-CIFAR,0x976EA74026E726554dB657fA54763abd0C3a0aa9,8e96cfd7-6a91-1cef-72e9-21c206c314f9,honest,True,False,...,"{5: 0.03267195767195767, 1: 0.0, 2: 0.0, 3: 0....",0.200000,0.416667,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",4.658730,0.0,0.000000,0,0.000000,0.0
19,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Cross Mal-MNIST Honest-CIFAR,0x14dC79964da2C08b23698B3D3cc7Ca32193d9955,b7fd4edd-361d-6858-0a0e-37ce12be516c,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.833333,"{5: 0.8333333333333334, 1: 0.0, 2: 0.0, 3: 0.0...",0.000000,0.0,0.000000,0,0.000000,0.0
20,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Malicious Both,0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f,31c4d68b-1f18-071e-dd3b-499b715548f9,malicious,False,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.833333,"{5: 0.8333333333333334, 1: 0.0, 2: 0.0, 3: 0.0...",0.000000,0.0,0.000000,0,0.000000,0.0
21,1,cifar-10,6,9f6a32eba0d11d8ee258e2e07895d654835c514ab0fd4e...,Free-rider Both,0xa0Ee7A142d267C1f36714E4a8F75612F20a79720,34eac3e3-3c1c-9ae8-841a-94fcb2cf42e5,freerider,True,False,...,"{5: 0.0, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 6: 0....",0.000000,0.416667,"{5: 0.4166666666666667, 1: 0.0, 2: 0.0, 3: 0.0...",0.000000,0.0,0.000000,0,0.000000,0.0


## Jupyter notes

Display all columns:

```python
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
```

Iterate all tasks with run data:

```python
for task_index, dataset, rd in session.iter_run_data():
    global_df = rd.get("global", pd.DataFrame())
    ...
```